# 7.1_Embedding_Model_Evaluation.ipynb

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

## Step 2: Load Inference Results Dataset and Configuration

In [2]:
from datasets import load_dataset
import torch

In [3]:
ds_test = load_dataset("dinushiTJ/nz_research_commons_inference_results", split="train")
display(ds_test)

README.md:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse'],
    num_rows: 3318
})

In [4]:
TASK_NAME = "Classification"
CONTEXT_LEN = 2048  # Gemma context length
EMBED_DIM = 768     # Gemma embedding size
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
# Install sentence-transformers if not already installed
%pip install -qU sentence-transformers

from sentence_transformers import SentenceTransformer
import huggingface_hub

# Log in to Hugging Face Hub to access gated models
huggingface_hub.login()

model = SentenceTransformer("google/embeddinggemma-300M")
print("EmbeddingGemma model loaded.")

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

EmbeddingGemma model loaded.


In [6]:
label_names = list(sorted(set(ds_test["classification_label"])))

In [7]:
label_names

[0, 1]

In [8]:
print(ds_test.features)

{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': ClassLabel(names=['maori_origin', 'non_maori_origin']), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'BaseEmbGemZeroLabel': Value('string'), 'BaseEmbGemZeroID': Value('int64'), 'BaseEmbGemZeroScore': Value('float64'), 'Base4BResponse': Value('string')}


In [9]:
label_feature = ds_test.features["classification_label"]

class_names = label_feature.names

print(class_names)

for i in range(len(class_names)):
    print(i, "->", label_feature.int2str(i))

['maori_origin', 'non_maori_origin']
0 -> maori_origin
1 -> non_maori_origin


In [10]:
print("Unique classification labels:", class_names)

Unique classification labels: ['maori_origin', 'non_maori_origin']


In [11]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer("google/embeddinggemma-300M").to(DEVICE)
base_model.eval()



Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (4): Normalize({})
)

In [12]:
base_label_embeddings = base_model.encode(
    class_names,
    prompt_name=TASK_NAME,
    convert_to_numpy=True,
    normalize_embeddings=True
)

In [13]:
base_label_embeddings

array([[-0.19262552, -0.03625868,  0.02727364, ..., -0.03909069,
        -0.01297866, -0.00118839],
       [-0.2096161 , -0.04762119,  0.01081607, ..., -0.02479461,
        -0.02834249, -0.00602297]], dtype=float32)

In [14]:
base_label_embeddings.shape

(2, 768)

In [15]:
def embed_document(text: str, token_count: int, model) -> np.ndarray:
    if token_count <= CONTEXT_LEN:
        emb = model.encode(
            text,
            prompt_name=TASK_NAME,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        return emb

    # Chunking path
    words = text.split()
    chunk_size = max(1, int(len(words) * CONTEXT_LEN / token_count))

    chunk_embeddings = []

    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i + chunk_size])
        emb = model.encode(
            chunk_text,
            prompt_name=TASK_NAME,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        chunk_embeddings.append(emb)

    # Mean pool chunk embeddings
    return np.mean(np.stack(chunk_embeddings, axis=0), axis=0)


In [16]:
def zero_shot_classify_base(example):

    doc_embedding = embed_document(
        text=example["embedding_text"],
        token_count=example["embedding_token_count"],
        model=base_model,
    )

    similarities = np.dot(
        doc_embedding,
        base_label_embeddings.T
    )

    best_idx = int(np.argmax(similarities))

    return {
        "BaseEmbGemZeroLabel": class_names[best_idx],
        "BaseEmbGemZeroID": best_idx,
        "BaseEmbGemZeroScore": float(similarities[best_idx])
    }

Now, let's apply the `zero_shot_classify_base` function to the test dataset (`ds_test`) to get the predictions from the base embedding model.

In [17]:
ds_test

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse'],
    num_rows: 3318
})

In [18]:
ds_test = ds_test.map(zero_shot_classify_base, batched=False)

Map:   0%|          | 0/3318 [00:00<?, ? examples/s]

In [19]:
print(ds_test.features)

{'title': Value('string'), 'authors': Value('string'), 'subjects': Value('string'), 'abstract': Value('string'), 'text': Value('string'), 'record_id_hash': Value('string'), 'prompt': Value('string'), 'token_count': Value('int32'), 'classification_label': ClassLabel(names=['maori_origin', 'non_maori_origin']), 'classification_reason': Value('string'), 'year': Value('string'), 'classification_confidence': Value('float32'), 'embedding_text': Value('string'), 'embedding_token_count': Value('int64'), 'BaseEmbGemZeroLabel': Value('string'), 'BaseEmbGemZeroID': Value('int64'), 'BaseEmbGemZeroScore': Value('float64'), 'Base4BResponse': Value('string')}


In [20]:
for i in range(5):
    print(
        label_feature.int2str(ds_test[i]["classification_label"]),
        "->",
        ds_test[i]["BaseEmbGemZeroLabel"]
    )


non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin
non_maori_origin -> non_maori_origin


### Push the dataset to Hugging Face Hub

Now, let's push the modified `ds_test` dataset to the Hugging Face Hub. You will be prompted to enter a name for your dataset.

In [21]:
repo_id = "dinushiTJ/nz_research_commons_inference_results"
ds_test.push_to_hub(repo_id, split='train')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  31%|###       | 5.12MB / 16.7MB            

CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz_research_commons_inference_results/commit/1b6f5f91d971bf1becd38cc591ff88bfb1f04d28', commit_message='Upload dataset', commit_description='', oid='1b6f5f91d971bf1becd38cc591ff88bfb1f04d28', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz_research_commons_inference_results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz_research_commons_inference_results'), pr_revision=None, pr_num=None)

End